# 制作间：合并 → 渲染 → 语音 → 视频

流水线里需要跑脚本的几步（30 合并、40 渲染、60 语音、70 视频）都在这里完成。内容步骤（00–20、31、50）由各个 agent 写进 Google Drive 的期刊文件夹，这里直接读。

用法：改第 1 格的两个路径和期号 → 菜单「代码执行程序 → 全部运行」。语音一步要 GPU：「代码执行程序 → 更改运行时类型 → T4 GPU」。

In [ ]:
# 1. 设置
DATA = "/content/drive/MyDrive/小满早知道"   # 数据目录（Drive 里的文件夹）
ISSUE = "004"                                # 期号，三位数
RUN_AUDIO = True                             # 没写讲读稿或不需要视频时改成 False

from google.colab import drive
drive.mount("/content/drive")
import os
ISSUE_DIR = f"{DATA}/issues/{ISSUE}"
assert os.path.isdir(ISSUE_DIR), f"找不到 {ISSUE_DIR}"
print(ISSUE_DIR)

In [ ]:
%%bash
# 2. 准备工具：引擎代码、Node 22、Chrome、中文字体
set -e
if [ -d /content/engine ]; then git -C /content/engine pull -q; else git clone -q --depth 1 https://github.com/jevejoze/kids-magazine-kit /content/engine; fi
if ! node -v 2>/dev/null | grep -q '^v2[2-9]'; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | bash - >/dev/null && apt-get install -y -qq nodejs >/dev/null
fi
apt-get install -y -qq fonts-noto-cjk ffmpeg >/dev/null
pip -q install playwright >/dev/null && python -m playwright install --with-deps chromium >/dev/null
node -v

In [ ]:
# 3. 看进度、检查已完成的步骤
import glob, subprocess
CHROME = sorted(glob.glob("/root/.cache/ms-playwright/chromium-*/chrome-linux/chrome"))[-1]
os.environ["CHROME"] = CHROME
!node /content/engine/scripts/pipeline.mjs status "$ISSUE_DIR"
!node /content/engine/scripts/pipeline.mjs check "$ISSUE_DIR" || true

In [ ]:
# 4. 第 30 步：合并整期
!node /content/engine/scripts/pipeline.mjs assemble "$ISSUE_DIR"

In [ ]:
# 5. 第 40 步：渲染 7 张图和 PDF（需要 31-review.md 结论为"通过"，这里只提醒不拦截）
review = f"{ISSUE_DIR}/31-review.md"
if not os.path.exists(review) or "结论：通过" not in open(review, encoding="utf-8").read():
    print("⚠️ 审稿还没通过，先渲染出来看看")
!node /content/engine/scripts/render.mjs "$ISSUE_DIR/30-issue.json" "$ISSUE_DIR/40-cards" --flat

## 第 60 步：语音

下面以 IndexTTS 为例。换成别的语音服务时，只要改写 `synth()`：把一句文字按指定角色念出来，存成 wav。

参考音放在 `数据目录/voices/`，`voices.json` 写每个角色用哪段，例如：

```json
{"narrator/zh": "narrator.wav", "dog/zh": "dog.wav", "mouse/en": "mouse.wav", "mouse/zh": "mouse.wav",
 "owl/zh": "owl.wav", "owl/en": "owl.wav", "raccoon/zh": "raccoon.wav", "raccoon/en": "raccoon.wav",
 "narrator/en": "narrator.wav"}
```

In [ ]:
# 6. 安装并加载 IndexTTS（按你自己的部署方式调整；只需运行一次）
if RUN_AUDIO:
    !if [ ! -d /content/index-tts ]; then git clone -q https://github.com/index-tts/index-tts /content/index-tts; fi
    # 按 index-tts 仓库 README 安装依赖、下载模型到 /content/index-tts/checkpoints
    import sys; sys.path.insert(0, "/content/index-tts")
    from indextts.infer_v2 import IndexTTS2
    tts = IndexTTS2(cfg_path="/content/index-tts/checkpoints/config.yaml", model_dir="/content/index-tts/checkpoints")

    EMO = {"开心": "开心、轻快", "兴奋": "非常兴奋", "惊讶": "很惊讶", "神秘": "压低声音、神秘", "紧张": "紧张",
           "困倦": "困倦、打哈欠", "得意": "得意洋洋", "委屈": "有点委屈", "温柔": "温柔", "平静": "平静"}

    def synth(text, ref_wav, out_wav, emotion=None):
        if emotion and emotion != "平静":
            tts.infer(spk_audio_prompt=ref_wav, text=text, output_path=out_wav,
                      use_emo_text=True, emo_text=EMO.get(emotion, emotion), emo_alpha=0.6)
        else:
            tts.infer(spk_audio_prompt=ref_wav, text=text, output_path=out_wav)

In [ ]:
# 7. 第 60 步：逐句合成（已存在的文件跳过，改了哪句就删掉对应的 wav 再跑）
import json, subprocess
if RUN_AUDIO:
    nar = json.load(open(f"{ISSUE_DIR}/50-narration.json", encoding="utf-8"))["narration"]
    voices = json.load(open(f"{DATA}/voices/voices.json", encoding="utf-8"))
    out = f"{ISSUE_DIR}/60-audio"; os.makedirs(out, exist_ok=True)
    files = []
    for page in nar:
        for i, line in enumerate(page["lines"]):
            f = f"{out}/c{page['card']}_{i:02d}.wav"
            if not os.path.exists(f):
                ref = f"{DATA}/voices/" + voices[f"{line['who']}/{line['lang']}"]
                synth(line["text"], ref, f, line.get("emotion"))
            sec = float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", f],
                                       capture_output=True, text=True).stdout)
            files.append({"card": page["card"], "index": i, "file": os.path.basename(f), "seconds": round(sec, 2)})
            print(f"c{page['card']}_{i:02d}  {sec:4.1f}s  {line['text'][:30]}")
    json.dump({"_meta": {"stage": "60-audio", "by": "制作间", "at": __import__("datetime").datetime.now().isoformat()},
               "voices": voices, "files": files}, open(f"{out}/manifest.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

In [ ]:
# 8. 第 70 步：合成讲读视频，并把 PDF 放进 70-final
if RUN_AUDIO:
    !python3 /content/engine/scripts/make_video.py "$ISSUE_DIR"
else:
    import shutil; os.makedirs(f"{ISSUE_DIR}/70-final", exist_ok=True)
    for p in glob.glob(f"{ISSUE_DIR}/40-cards/*.pdf"): shutil.copy(p, f"{ISSUE_DIR}/70-final")
!ls -la "$ISSUE_DIR/70-final"